# BioTek EL406

The BioTek EL406 plate washer has four subsystems — manifold, syringe pump, peristaltic pump, and shaker — each exposed as a capability on the `EL406` device.

## Setup

In [ ]:
import os

from pylabrobot.agilent.biotek.el406 import EL406
from pylabrobot.resources import Cor_96_wellplate_360ul_Fb

os.environ["DYLD_LIBRARY_PATH"] = "/opt/homebrew/lib:" + os.environ.get("DYLD_LIBRARY_PATH", "")
el406 = EL406(name="el406")
await el406.setup()

plate = Cor_96_wellplate_360ul_Fb(name="plate")
el406.plate_holder.assign_child_resource(plate)

## Manifold (plate washing)

The wash manifold is the primary fluid system. A basic wash with default settings:

In [ ]:
await el406.washer.wash(cycles=3, dispense_volume=300)

Use {class}`~pylabrobot.agilent.biotek.el406.plate_washing_backend.EL406PlateWasher96Backend.WashParams` to configure buffer, soak, and shake options:

In [ ]:
from pylabrobot.agilent.biotek.el406.plate_washing_backend import EL406PlateWasher96Backend

await el406.washer.wash(
  cycles=3,
  dispense_volume=300,
  backend_params=EL406PlateWasher96Backend.WashParams(
    buffer="A",
    soak_duration=10,
    shake_duration=5,
    shake_intensity="Medium",
  ),
)

Aspirate and dispense:

In [ ]:
await el406.washer.aspirate()
await el406.washer.dispense(volume=200)

Prime the manifold lines using {class}`~pylabrobot.agilent.biotek.el406.plate_washing_backend.EL406PlateWasher96Backend.PrimeParams`:

In [ ]:
await el406.washer.prime(
  backend_params=EL406PlateWasher96Backend.PrimeParams(
    volume=10000, buffer="A",
  ),
)

## Syringe pump

Precise low-volume dispensing via dual syringe pumps (A/B). Prime the syringe lines first:

In [ ]:
from pylabrobot.agilent.biotek.el406.syringe_dispensing_backend8 import EL406SyringeDispensingBackend8

await el406.syringe_dispenser.prime(
  volume=5000,
  backend_params=EL406SyringeDispensingBackend8.PrimeParams(
    syringe="A"
  )
)

Dispense the same volume to all columns:

In [ ]:
await el406.syringe_dispenser.dispense(volumes=50)

Different volumes per column:

In [ ]:
await el406.syringe_dispenser.dispense(volumes={
  **{c: 100 for c in range(1, 7)},
  **{c: 50 for c in range(7, 13)},
})

## Peristaltic pump

Continuous-flow dispensing with cassette selection and row/column masking. Prime the lines first:

In [ ]:
from pylabrobot.agilent.biotek.el406.peristaltic_dispensing_backend8 import EL406PeristalticDispensingBackend8

await el406.peristaltic_dispenser.prime(
  volume=300,
  backend_params=EL406PeristalticDispensingBackend8.PrimeParams(
    flow_rate="High"
  )
)

Dispense the same volume to all columns:

In [ ]:
await el406.peristaltic_dispenser.dispense(volumes=100)

Different volumes per column:

In [ ]:
await el406.peristaltic_dispenser.dispense(volumes={
  **{c: 200 for c in range(1, 7)},
  **{c: 100 for c in range(7, 13)},
})

Purge the lines after dispensing:

In [ ]:
await el406.peristaltic_dispenser.purge(
  volume=300,
  backend_params=EL406PeristalticDispensingBackend8.PrimeParams(
    flow_rate="High"
  )
)

## Shaking

The generic `Shaker.shake(speed, duration)` interface works — the EL406 reads the plate from the plate holder automatically. The `speed` parameter is ignored since the EL406 uses discrete intensity levels.

In [ ]:
await el406.shaker.shake(speed=0, duration=10)

Use {class}`~pylabrobot.agilent.biotek.el406.shaking_backend.EL406ShakingBackend.ShakeParams` to control intensity and soak duration:

In [ ]:
from pylabrobot.agilent.biotek.el406.shaking_backend import EL406ShakingBackend

await el406.shaker.shake(
  speed=0,
  duration=10,
  backend_params=EL406ShakingBackend.ShakeParams(
    intensity="Fast",
    soak_duration=10,
  ),
)

## Advanced usage

### Manual batching

Each step command (wash, dispense, prime, etc.) automatically starts and stops a **batch** — the device's "ready to execute" mode. When you call multiple commands in sequence, each one opens and closes its own batch, which adds overhead.

To avoid this, wrap multiple commands in a single `driver.batch()` context manager. The inner commands detect they are already inside a batch and skip the redundant start/cleanup:

```{note}
When dispensing different volumes per column (e.g. `volumes={1: 100, 7: 50}`), the `dispense()` method handles batching internally — it groups columns by volume and sends one command per group, all within a single batch. You only need manual batching when combining **different operations** (e.g. prime + dispense + shake).
```

In [ ]:
async with el406.driver.batch():
    await el406.peristaltic_dispenser.prime(
        volume=300,
        backend_params=EL406PeristalticDispensingBackend8.PrimeParams(flow_rate="High"),
    )
    await el406.peristaltic_dispenser.dispense(volumes=100)
    await el406.shaker.shake(speed=0, duration=5)

### Combining subsystems in a single batch

A full protocol often chains operations across subsystems. Manual batching keeps the device in its ready state throughout:

In [ ]:
async with el406.driver.batch():
    # Step 1: wash the plate
    await el406.washer.wash(cycles=3, dispense_volume=300)

    # Step 2: dispense reagent via syringe
    await el406.syringe_dispenser.dispense(volumes=25)

    # Step 3: shake to mix
    await el406.shaker.shake(
        speed=0,
        duration=30,
        backend_params=EL406ShakingBackend.ShakeParams(intensity="Medium"),
    )

    # Step 4: final aspirate
    await el406.washer.aspirate()

### Column and row masking

Target specific wells using column dicts and the `rows` parameter. For 384-well plates, rows are grouped into 2 quadrants; for 1536-well, into 4:

In [ ]:
# Dispense 200 uL to columns 1-6 only
await el406.peristaltic_dispenser.dispense(
    volumes={c: 200 for c in range(1, 7)},
)

# On a 384-well plate, dispense to row quadrant 1 only
await el406.peristaltic_dispenser.dispense(
    volumes=100,
    backend_params=EL406PeristalticDispensingBackend8.DispenseParams(
        rows=[1],
    ),
)

## Device-level operations

In [ ]:
from pylabrobot.agilent.biotek.el406.enums import EL406WasherManifold

await el406.driver.set_washer_manifold(EL406WasherManifold.TUBE_96_DUAL)
await el406.driver.reset()

## Teardown

In [ ]:
await el406.stop()